# 🔵 Phase 2 — BNS Model Training
**Bharatiya Nyaya Sanhita (BNS) 2023 — FIR Section Prediction**

This notebook trains a new ML model on BNS sections (358 classes) replacing the old IPC model (511 classes).

### Pipeline
1. Load & verify datasets
2. Preprocess FIR text
3. Encode labels (358 BNS classes)
4. TF-IDF vectorization with `ngram_range=(1,2)`
5. Train model (LogisticRegression + LinearSVC)
6. Evaluate & compare
7. Save `bns_model.pkl` + `tfidf_vectorizer.pkl` + `label_encoder.pkl`

## 📦 Step 1 — Install & Import

In [ ]:
# Install required packages if not already available
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'joblib', '-q'])
print('✅ All packages ready')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, top_k_accuracy_score
)

print('✅ Imports successful')
print(f'scikit-learn version: {__import__("sklearn").__version__}')

## 📂 Step 2 — Load Datasets

In [ ]:
# ── CHANGE THESE PATHS if your files are elsewhere ──────────────────
FIR_DATASET_PATH  = 'fir_dataset_with_id.csv'   # columns: fir_id, fir_text, bns_section
BNS_SECTIONS_PATH = 'bns_sections.csv'           # columns: Chapter, Chapter_name, Section, Section _name, Description
# ─────────────────────────────────────────────────────────────────────

fir_df  = pd.read_csv(FIR_DATASET_PATH)
bns_df  = pd.read_csv(BNS_SECTIONS_PATH)

print('=== FIR Dataset ===')
print(f'Shape : {fir_df.shape}')
print(f'Columns: {fir_df.columns.tolist()}')
print(fir_df.head(3))
print()
print('=== BNS Sections ===')
print(f'Shape : {bns_df.shape}')
print(f'Columns: {bns_df.columns.tolist()}')
print(bns_df.head(3))

In [ ]:
# ── Data health check ────────────────────────────────────────────────
print('📊 FIR Dataset Summary')
print(f"  Total FIRs         : {len(fir_df):,}")
print(f"  Unique BNS sections: {fir_df['bns_section'].nunique()}")
print(f"  Null values        : {fir_df.isnull().sum().to_dict()}")
print()
print('📋 BNS Sections Summary')
print(f"  Total sections: {len(bns_df)}")
print(f"  Chapters      : {bns_df['Chapter'].nunique()}")
print()

# Check coverage
all_bns  = set(bns_df['Section'].tolist())
fir_secs = set(fir_df['bns_section'].tolist())
missing  = sorted(all_bns - fir_secs)
print(f"  BNS sections with FIR training data : {len(fir_secs)}")
print(f"  BNS sections WITHOUT training data  : {len(missing)}")
if missing:
    print(f"  Missing sections: {missing}")
    print("  ⚠️  Sections 1-45 & 358 are definitional/procedural — no FIR cases expected.")

## 🧹 Step 3 — Text Preprocessing

In [ ]:
def preprocess_text(text: str) -> str:
    """
    Clean and normalise raw FIR complaint text.
    - Lowercase
    - Remove special characters / extra whitespace
    - Keep alphanumeric + spaces
    """
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)   # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()    # collapse whitespace
    return text


# Apply preprocessing
fir_df['clean_text'] = fir_df['fir_text'].apply(preprocess_text)

# Remove empty rows (after cleaning)
before = len(fir_df)
fir_df = fir_df[fir_df['clean_text'].str.len() > 5].reset_index(drop=True)
print(f'✅ Preprocessing done  |  Before: {before}  →  After: {len(fir_df)}')
print()
print('Sample (raw → cleaned):')
for i in [0, 1, 2]:
    print(f"  [{i}] RAW    : {fir_df['fir_text'].iloc[i]}")
    print(f"       CLEANED: {fir_df['clean_text'].iloc[i]}")
    print()

## 🏷️ Step 4 — Label Encoding (312 active BNS classes)

In [ ]:
# Encode bns_section integers → contiguous label indices
le = LabelEncoder()
fir_df['label'] = le.fit_transform(fir_df['bns_section'])

NUM_CLASSES = len(le.classes_)

print(f'✅ Label encoder fitted')
print(f'   Number of classes : {NUM_CLASSES}')
print(f'   Classes (first 20): {le.classes_[:20].tolist()}')
print(f'   Classes (last 10) : {le.classes_[-10:].tolist()}')
print()

# Class distribution
dist = fir_df['bns_section'].value_counts()
print(f'   Min samples per class: {dist.min()}')
print(f'   Max samples per class: {dist.max()}')
print(f'   Mean samples/class   : {dist.mean():.1f}')

In [ ]:
# Visualise class distribution
plt.figure(figsize=(16, 4))
plt.bar(range(len(dist)), dist.values, color='steelblue', alpha=0.7)
plt.xlabel('BNS Section (sorted by frequency)')
plt.ylabel('Number of FIR samples')
plt.title('Training Sample Distribution Across BNS Sections')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120)
plt.show()
print('📊 Distribution chart saved as class_distribution.png')

## ✂️ Step 5 — Train / Test Split

In [ ]:
X = fir_df['clean_text'].values
y = fir_df['label'].values

# Stratified split → 80% train, 20% test
# Use stratify=y to keep class balance in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'✅ Train/Test split done')
print(f'   Training samples : {len(X_train):,}  ({len(X_train)/len(X)*100:.1f}%)')
print(f'   Test samples     : {len(X_test):,}  ({len(X_test)/len(X)*100:.1f}%)')
print(f'   Total            : {len(X):,}')

## 🔠 Step 6 — TF-IDF Vectorizer with `ngram_range=(1,2)`

In [ ]:
# ── TF-IDF settings (as required by Phase 2 spec) ───────────────────
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),      # unigrams + bigrams (required)
    max_features=50_000,     # vocabulary size cap
    sublinear_tf=True,       # apply log(1+tf) — better for text
    min_df=1,                # keep rare terms (small dataset)
    analyzer='word',
    token_pattern=r'\b[a-z][a-z0-9]*\b'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'✅ TF-IDF fitted')
print(f'   Vocabulary size   : {len(tfidf.vocabulary_):,}')
print(f'   Train matrix shape: {X_train_tfidf.shape}')
print(f'   Test  matrix shape: {X_test_tfidf.shape}')
print(f'   ngram_range       : {tfidf.ngram_range}')

## 🤖 Step 7 — Train Models

In [ ]:
# ── Model A : Logistic Regression (gives probability scores) ─────────
print('Training Logistic Regression...')
lr_model = LogisticRegression(
    C=5.0,
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
lr_model.fit(X_train_tfidf, y_train)
lr_acc = accuracy_score(y_test, lr_model.predict(X_test_tfidf))
print(f'  ✅ Logistic Regression Accuracy : {lr_acc*100:.2f}%')

In [ ]:
# ── Model B : LinearSVC (fast, usually higher accuracy) ──────────────
print('Training LinearSVC...')
svc_model = LinearSVC(
    C=1.0,
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)
svc_model.fit(X_train_tfidf, y_train)
svc_acc = accuracy_score(y_test, svc_model.predict(X_test_tfidf))
print(f'  ✅ LinearSVC Accuracy           : {svc_acc*100:.2f}%')

print()
print('📊 Model Comparison')
print(f'   Logistic Regression: {lr_acc*100:.2f}%')
print(f'   LinearSVC          : {svc_acc*100:.2f}%')

# Pick the winner
if lr_acc >= svc_acc:
    best_model = lr_model
    best_name  = 'Logistic Regression'
else:
    best_model = svc_model
    best_name  = 'LinearSVC'

print(f'\n🏆 Best model: {best_name}  ({max(lr_acc, svc_acc)*100:.2f}%)')

## 📈 Step 8 — Evaluation

In [ ]:
# ── Detailed classification report ───────────────────────────────────
y_pred = best_model.predict(X_test_tfidf)

# Map numeric labels back to BNS section numbers for readable report
target_names = [f'BNS-{int(c)}' for c in le.classes_]

print(f'=== Classification Report ({best_name}) ===')
print(classification_report(
    y_test, y_pred,
    target_names=target_names,
    zero_division=0
))

In [ ]:
# ── Top-3 Accuracy (important for IO predict page) ───────────────────
# LinearSVC doesn't give probabilities natively — use LR for top-k
y_prob_lr = lr_model.predict_proba(X_test_tfidf)

top1 = accuracy_score(y_test, lr_model.predict(X_test_tfidf))
top3 = top_k_accuracy_score(y_test, y_prob_lr, k=3)
top5 = top_k_accuracy_score(y_test, y_prob_lr, k=5)

print('📊 Top-K Accuracy (Logistic Regression — used for ranking predictions)')
print(f'   Top-1 : {top1*100:.2f}%')
print(f'   Top-3 : {top3*100:.2f}%')
print(f'   Top-5 : {top5*100:.2f}%')
print()
print('ℹ️  Top-3/5 are the relevant metrics for the "Similar Cases" and IO predict page.')

In [ ]:
# ── Per-class accuracy summary ────────────────────────────────────────
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)

class_report_df = pd.DataFrame({
    'BNS_Section': le.classes_,
    'Precision'  : precision.round(3),
    'Recall'     : recall.round(3),
    'F1_Score'   : f1.round(3),
    'Support'    : support
})

print('Top 15 sections by F1 score:')
print(class_report_df.sort_values('F1_Score', ascending=False).head(15).to_string(index=False))
print()
print('Bottom 10 sections by F1 score (may need more training data):')
print(class_report_df[class_report_df['Support'] > 0].sort_values('F1_Score').head(10).to_string(index=False))

In [ ]:
# ── Macro summary bar chart ────────────────────────────────────────────
metrics_summary = {
    'Top-1 Accuracy': top1,
    'Top-3 Accuracy': top3,
    'Top-5 Accuracy': top5,
    'Macro F1'      : f1.mean()
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(metrics_summary.keys(), [v*100 for v in metrics_summary.values()],
              color=['#2196F3','#4CAF50','#FF9800','#9C27B0'], alpha=0.85, edgecolor='white')
ax.set_ylabel('Score (%)')
ax.set_title(f'BNS Model Performance Summary — {best_name}\n({NUM_CLASSES} classes | {len(X_train):,} training samples)')
ax.set_ylim(0, 110)
for bar, val in zip(bars, metrics_summary.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val*100:.1f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('model_performance.png', dpi=120)
plt.show()
print('📊 Performance chart saved as model_performance.png')

## 💾 Step 9 — Save Models (bns_model.pkl, tfidf_vectorizer.pkl, label_encoder.pkl)

In [ ]:
import os

# ── Save all artefacts ────────────────────────────────────────────────
SAVE_DIR = '.'   # change to your project path, e.g. 'ai_module/models/'

tfidf_path   = os.path.join(SAVE_DIR, 'tfidf_vectorizer.pkl')
model_path   = os.path.join(SAVE_DIR, 'bns_model.pkl')
encoder_path = os.path.join(SAVE_DIR, 'label_encoder.pkl')
lr_path      = os.path.join(SAVE_DIR, 'bns_lr_model.pkl')   # always save LR for probabilities

joblib.dump(tfidf,      tfidf_path)
joblib.dump(best_model, model_path)
joblib.dump(le,         encoder_path)
joblib.dump(lr_model,   lr_path)     # needed for Top-K / confidence scores

print('✅ Saved files:')
for path in [tfidf_path, model_path, encoder_path, lr_path]:
    size = os.path.getsize(path) / 1024
    print(f'   📁 {path}  ({size:.1f} KB)')

print()
print('🔵 Phase 2 COMPLETE — update app.py to load these 3 files:')
print('   tfidf_vectorizer.pkl  →  tfidf = joblib.load("tfidf_vectorizer.pkl")')
print('   bns_model.pkl         →  model = joblib.load("bns_model.pkl")')
print('   label_encoder.pkl     →  le    = joblib.load("label_encoder.pkl")')

## ✅ Step 10 — Quick Sanity Check (Predict on New Text)

In [ ]:
def predict_bns_section(text: str, top_k: int = 5) -> pd.DataFrame:
    """
    Predict BNS section for a given FIR complaint text.
    Returns top-K predictions with confidence scores.
    Uses Logistic Regression for probability estimates.
    """
    cleaned  = preprocess_text(text)
    vec      = tfidf.transform([cleaned])
    probs    = lr_model.predict_proba(vec)[0]           # shape: (num_classes,)
    top_idx  = np.argsort(probs)[::-1][:top_k]

    results = pd.DataFrame({
        'Rank'        : range(1, top_k + 1),
        'BNS_Section' : le.inverse_transform(top_idx),
        'Confidence'  : (probs[top_idx] * 100).round(2)
    })
    return results


# ── Test Cases ────────────────────────────────────────────────────────
test_cases = [
    "The accused murdered my brother with a sharp weapon during a property dispute.",
    "My employer has not paid my salary for 3 months and threatened me when I asked.",
    "The accused broke into my house at night and stole gold jewellery and cash.",
    "The accused stalked me online for months, sent obscene messages despite my refusal.",
    "The accused forged my signature on property documents and illegally transferred my land.",
    "My husband and his family are harassing me daily for more dowry.",
    "A group of men attacked and robbed me at knifepoint near the railway station."
]

for i, tc in enumerate(test_cases, 1):
    print(f'\n🔍 Test Case {i}:')
    print(f'   "{tc[:80]}..."' if len(tc) > 80 else f'   "{tc}"')
    result = predict_bns_section(tc, top_k=3)
    for _, row in result.iterrows():
        print(f'   #{int(row["Rank"])} → BNS Section {int(row["BNS_Section"])}  ({row["Confidence"]}% confidence)')

## 📋 Step 11 — Model Metadata File (for app.py reference)

In [ ]:
import json, datetime

metadata = {
    "model_version"     : "2.0",
    "law"               : "Bharatiya Nyaya Sanhita (BNS) 2023",
    "trained_on"        : str(datetime.date.today()),
    "num_classes"       : int(NUM_CLASSES),
    "total_samples"     : int(len(fir_df)),
    "train_samples"     : int(len(X_train)),
    "test_samples"      : int(len(X_test)),
    "best_model"        : best_name,
    "top1_accuracy"     : round(float(top1), 4),
    "top3_accuracy"     : round(float(top3), 4),
    "top5_accuracy"     : round(float(top5), 4),
    "macro_f1"          : round(float(f1.mean()), 4),
    "tfidf_settings"    : {
        "ngram_range"   : [1, 2],
        "max_features"  : 50000,
        "sublinear_tf"  : True
    },
    "files": {
        "tfidf"         : "tfidf_vectorizer.pkl",
        "best_model"    : "bns_model.pkl",
        "lr_model"      : "bns_lr_model.pkl",
        "label_encoder" : "label_encoder.pkl"
    }
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ model_metadata.json saved')
print(json.dumps(metadata, indent=2))

## 📌 app.py Integration Snippet
Copy this into your `app.py` to replace the old IPC model loading code.

In [ ]:
snippet = '''
# ── Phase 2: BNS Model Loading (paste into app.py) ──────────────────────────
import joblib, re, numpy as np

tfidf   = joblib.load('tfidf_vectorizer.pkl')
model   = joblib.load('bns_model.pkl')          # best model (LR or SVC)
lr_model = joblib.load('bns_lr_model.pkl')      # always LR for probabilities
le      = joblib.load('label_encoder.pkl')

NUM_CLASSES = len(le.classes_)   # 312 (active) or 358 (all BNS)


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text


def predict_bns(text, top_k=5):
    """
    Returns list of (bns_section_number, confidence_pct) tuples.
    Use this for the IO Predict page and Similar Cases feature.
    """
    cleaned = preprocess_text(text)
    vec     = tfidf.transform([cleaned])
    probs   = lr_model.predict_proba(vec)[0]
    top_idx = np.argsort(probs)[::-1][:top_k]
    return [
        {
          "section"    : int(le.inverse_transform([idx])[0]),
          "confidence" : round(float(probs[idx]) * 100, 2)
        }
        for idx in top_idx
    ]

# Usage:
# results = predict_bns("Accused murdered my brother with a knife")
# → [{"section": 103, "confidence": 87.3}, {"section": 101, "confidence": 7.2}, ...]
# ─────────────────────────────────────────────────────────────────────────────
'''

with open('app_snippet.py', 'w') as f:
    f.write(snippet)

print(snippet)

---
## 🎉 Phase 2 Complete!

| File | Purpose |
|---|---|
| `tfidf_vectorizer.pkl` | TF-IDF vectorizer with `ngram_range=(1,2)` |
| `bns_model.pkl` | Best model (LR or SVC) |
| `bns_lr_model.pkl` | Logistic Regression (for confidence scores / Top-K) |
| `label_encoder.pkl` | Encodes BNS section numbers ↔ model indices |
| `model_metadata.json` | Version, accuracy, settings |

### Next: Phase 3 — Backend Logic in app.py
- Update file paths to new `.pkl` files  
- Add `is_valid_phone()` regex  
- Add `get_location_suggestions()` via Nominatim  
- Add `get_similar_firs()` cosine similarity